# 🚦 Gridlock Hackathon 2.0 — Traffic Demand Prediction
**Flipkart × HackerEarth | Bengaluru Smart Mobility Challenge**

**Goal:** Predict `demand` (traffic demand at a given location & timestamp)

**Strategy:** EDA → Feature Engineering → LightGBM + XGBoost Ensemble → Submit

##  Step 1: Import Libraries

In [ ]:
import os
import warnings
import logging

# ── Silence ALL warnings before importing heavy libs ──────────────────────────
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
for _name in ('lightgbm', 'xgboost', 'py.warnings', 'root'):
    logging.getLogger(_name).setLevel(logging.ERROR)

import pandas as pd
import numpy as np

import matplotlib
matplotlib.use('Agg')        # non-interactive backend — no red squiggle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

import xgboost as xgb
xgb.set_config(verbosity=0)

try:
    import pygeohash as pgh   # pip install pygeohash
    GEOHASH_AVAILABLE = True
except Exception:
    pgh = None
    GEOHASH_AVAILABLE = False

# ── Version check ─────────────────────────────────────────────────────────────
print('pandas   :', pd.__version__)
print('numpy    :', np.__version__)
print('lightgbm :', lgb.__version__)
print('xgboost  :', xgb.__version__)
print('pygeohash:', 'OK' if GEOHASH_AVAILABLE else 'not installed — skipping lat/lon')
print()
print('All libraries ready!')

##  Step 2: Load Data

In [ ]:
train      = pd.read_csv('dataset/dataset/train.csv')
test       = pd.read_csv('dataset/dataset/test.csv')
sample_sub = pd.read_csv('dataset/dataset/sample_submission.csv')

print(f'Train shape      : {train.shape}')
print(f'Test shape       : {test.shape}')
print(f'Sample submission: {sample_sub.shape}')
train.head()

##  Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print('=== Column Data Types ===')
print(train.dtypes)
print('\n=== Missing Values ===')
print(train.isnull().sum())
print('\n=== Target Variable (demand) Stats ===')
print(train['demand'].describe())

In [ ]:
# Unique values in categorical columns
for col in ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'day']:
    if col in train.columns:
        print(f'{col}: {list(train[col].unique())}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train['demand'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Demand Distribution', fontsize=13)
axes[0].set_xlabel('Demand')

axes[1].hist(np.log1p(train['demand']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Log(1+Demand) Distribution', fontsize=13)
axes[1].set_xlabel('Log Demand')

plt.tight_layout()
plt.savefig('demand_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart saved as demand_distribution.png')

##  Step 4: Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Geohash → lat / lon
    if GEOHASH_AVAILABLE and pgh is not None and 'geohash' in df.columns:
        try:
            decoded   = df['geohash'].apply(lambda x: pgh.decode(str(x)) if pd.notnull(x) else (np.nan, np.nan))
            df['lat'] = decoded.apply(lambda x: x[0])
            df['lon'] = decoded.apply(lambda x: x[1])
        except Exception:
            df['lat'] = 0.0
            df['lon'] = 0.0

    # Geohash prefix clusters
    if 'geohash' in df.columns:
        df['geohash_4'] = df['geohash'].str[:4]
        df['geohash_5'] = df['geohash'].str[:5]

    # Timestamp → time features  (format: H:MM)
    if 'timestamp' in df.columns:
        parts          = df['timestamp'].astype(str).str.split(':', expand=True)
        df['hour']     = parts[0].astype(float)
        df['minute']   = parts[1].astype(float)
        df['hour_minute']     = df['hour'] * 60 + df['minute']
        df['is_peak_morning'] = ((df['hour'] >= 7)  & (df['hour'] <= 10)).astype(int)
        df['is_peak_evening'] = ((df['hour'] >= 17) & (df['hour'] <= 20)).astype(int)
        df['is_night']        = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
        df['sin_hour']        = np.sin(2 * np.pi * df['hour'] / 24)
        df['cos_hour']        = np.cos(2 * np.pi * df['hour'] / 24)

    # Day cyclical
    if 'day' in df.columns:
        df['sin_day'] = np.sin(2 * np.pi * df['day'] / 7)
        df['cos_day'] = np.cos(2 * np.pi * df['day'] / 7)

    # Temperature as numeric
    if 'Temperature' in df.columns:
        df['Temperature'] = pd.to_numeric(df['Temperature'], errors='coerce')

    return df


train = engineer_features(train)
test  = engineer_features(test)
print(f'Train shape after FE : {train.shape}')
print(f'Test  shape after FE : {test.shape}')

##  Step 5: Encode Categorical Variables

In [ ]:
cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geohash_4', 'geohash_5']
cat_cols = [c for c in cat_cols if c in train.columns]

for col in cat_cols:
    le       = LabelEncoder()
    combined = pd.concat([
        train[col].astype(str).fillna('NaN'),
        test[col].astype(str).fillna('NaN')
    ])
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str).fillna('NaN'))
    test[col]  = le.transform(test[col].astype(str).fillna('NaN'))

print('Label encoding done for:', cat_cols)

##  Step 6: Prepare Features & Target

In [ ]:
exclude      = ['Index', 'demand', 'timestamp', 'geohash']
feature_cols = [
    c for c in train.columns
    if c not in exclude
    and train[c].dtype in [np.float64, np.int64, np.float32, np.int32]
]

print(f'Total features ({len(feature_cols)}): {feature_cols}')

X      = train[feature_cols].copy()
y      = train['demand'].copy()
X_test = test[feature_cols].copy()

# Fill NaN with column median
medians = X.median()
X       = X.fillna(medians)
X_test  = X_test.fillna(medians)

print(f'\nX shape     : {X.shape}  | NaN: {X.isnull().sum().sum()}')
print(f'X_test shape: {X_test.shape}  | NaN: {X_test.isnull().sum().sum()}')

##  Step 7: Train LightGBM (5-Fold CV)

In [ ]:
lgb_params = {
    'objective'        : 'regression',
    'metric'           : 'rmse',
    'learning_rate'    : 0.05,
    'num_leaves'       : 127,
    'max_depth'        : -1,
    'min_child_samples': 20,
    'feature_fraction' : 0.8,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

N_FOLDS    = 5
kf         = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
lgb_oof    = np.zeros(len(X))
lgb_preds  = np.zeros(len(X_test))
lgb_scores = []
lgb_model  = None

print(f'Training LightGBM with {N_FOLDS}-Fold CV...')

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    lgb_model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    val_pred         = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
    lgb_oof[val_idx] = val_pred
    lgb_preds       += lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration) / N_FOLDS

    fold_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    lgb_scores.append(fold_rmse)
    print(f'  Fold {fold+1}  RMSE: {fold_rmse:.5f}  |  Best iter: {lgb_model.best_iteration}')

print(f'\nLightGBM CV RMSE: {np.mean(lgb_scores):.5f} +/- {np.std(lgb_scores):.5f}')

##  Step 8: Train XGBoost (5-Fold CV)

In [ ]:
xgb_params = {
    'objective'        : 'reg:squarederror',
    'eval_metric'      : 'rmse',
    'learning_rate'    : 0.05,
    'max_depth'        : 7,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 1.0,
    'seed'             : 42,
    'nthread'          : -1,
    'verbosity'        : 0,
}

xgb_oof    = np.zeros(len(X))
xgb_preds  = np.zeros(len(X_test))
xgb_scores = []

print(f'Training XGBoost with {N_FOLDS}-Fold CV...')

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval   = xgb.DMatrix(X_val, label=y_val)
    dtest  = xgb.DMatrix(X_test)

    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=3000,
        evals=[(dval, 'val')],
        early_stopping_rounds=100,
        verbose_eval=False,
    )

    val_pred         = model.predict(dval)
    xgb_oof[val_idx] = val_pred
    xgb_preds       += model.predict(dtest) / N_FOLDS

    fold_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    xgb_scores.append(fold_rmse)
    print(f'  Fold {fold+1}  RMSE: {fold_rmse:.5f}  |  Best iter: {model.best_iteration}')

print(f'\nXGBoost CV RMSE: {np.mean(xgb_scores):.5f} +/- {np.std(xgb_scores):.5f}')

##  Step 9: Ensemble — LightGBM + XGBoost

In [ ]:
LGB_W = 0.6
XGB_W = 0.4

ensemble_oof   = LGB_W * lgb_oof   + XGB_W * xgb_oof
ensemble_preds = LGB_W * lgb_preds + XGB_W * xgb_preds
ensemble_preds = np.clip(ensemble_preds, 0.0, 1.0)

ens_rmse = np.sqrt(mean_squared_error(y, ensemble_oof))
lgb_rmse = np.sqrt(mean_squared_error(y, lgb_oof))
xgb_rmse = np.sqrt(mean_squared_error(y, xgb_oof))

print(f'LightGBM  OOF RMSE : {lgb_rmse:.5f}')
print(f'XGBoost   OOF RMSE : {xgb_rmse:.5f}')
print(f'Ensemble  OOF RMSE : {ens_rmse:.5f}  <-- best')
print(f'\nPrediction range: [{ensemble_preds.min():.5f}, {ensemble_preds.max():.5f}]  mean={ensemble_preds.mean():.5f}')

##  Step 10: Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature'   : feature_cols,
    'importance': lgb_model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(20), x='importance', y='feature', palette='viridis')
plt.title('Top 20 Feature Importances (LightGBM Gain)', fontsize=13)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print('Feature importance saved')
print(importance_df.head(10).to_string(index=False))

## Step 11: Generate Submission File

In [ ]:
submission = pd.DataFrame({
    'Index' : test['Index'],
    'demand': ensemble_preds,
})

submission.to_csv('submission.csv', index=False)

print('Submission saved -> submission.csv')
print(f'Rows: {len(submission):,}  |  Columns: {list(submission.columns)}')
print('\nFirst 5 predictions:')
print(submission.head().to_string(index=False))
print('\nExpected format (sample_submission):')
print(sample_sub.head().to_string(index=False))

## ✅ Done!
Upload `submission.csv` + this `.ipynb` on HackerEarth.

| Model | CV RMSE |
|---|---|
| LightGBM | ~0.03586 |
| XGBoost | ~0.03597 |
| **Ensemble** | **~0.03543** |